In [1]:


# ── Cell 1: Imports, Seed, Config ────────────────────────────
#
# PORTABLE sys.path BOOTSTRAP
# This notebook lives at clients/main_framework/MLP_depression*.ipynb
# → project root is two levels up from os.getcwd()
#
import sys, os, time, pickle, warnings

_HERE         = os.path.abspath(os.getcwd())            # .../clients/main_framework
_PROJECT_ROOT = os.path.dirname(os.path.dirname(_HERE)) # .../project
if _PROJECT_ROOT not in sys.path:
    sys.path.insert(0, _PROJECT_ROOT)

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning, module="opacus")

from utils.seed import fix_all_seeds
fix_all_seeds()

import pandas as pd
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import LabelEncoder
from imblearn.over_sampling import SMOTE
from opacus import PrivacyEngine

from config import *          # pulls BASE_PATH, RESULTS_PATH, AGGREGATED_GRAD_FILE, …
from utils.logger import log_round

# ── Client identity ───────────────────────────────────────────
# For hospital_1: CLIENT_ID = "hospital_1", DATA_FILE = "Final_Depression1_anonymised.csv"
# For hospital_2: CLIENT_ID = "hospital_2", DATA_FILE = "Final_Depression2_anonymised.csv"
# For hospital_3: CLIENT_ID = "hospital_3", DATA_FILE = "Final_Depression3_anonymised.csv"
CLIENT_ID = "hospital_1"
DATA_FILE = "Final_Depression1_anonymised.csv"

MODEL_FILE           = os.path.join(RESULTS_PATH, f"{CLIENT_ID}_local_model.pth")
GLOBAL_GRADIENT_FILE = AGGREGATED_GRAD_FILE   # ← from config, no hardcoding

print(f"[CONFIG] Client     : {CLIENT_ID}")
print(f"[CONFIG] Experiment : {EXPERIMENT_NAME}")
print(f"[CONFIG] DP enabled : {DP_ENABLED}")
print(f"[CONFIG] HE enabled : {HE_ENABLED}")
print(f"[CONFIG] Seed       : {SEED}")
print(f"[CONFIG] Project root: {BASE_PATH}")


[SEED] All seeds fixed to 456
[CONFIG] Client     : hospital_1
[CONFIG] Experiment : fl_anon_he
[CONFIG] DP enabled : False
[CONFIG] HE enabled : True
[CONFIG] Seed       : 456
[CONFIG] Project root: /Users/ravi/Desktop/Paper/hybrid-privacy-fl-healthcare


In [2]:
# ── Cell 2: Load Data ─────────────────────────────────────────
data_path = os.path.join(DATA_PATH, DATA_FILE)
df = pd.read_csv(data_path)
print(f"[DATA] Loaded: {data_path}")
print(f"[DATA] Shape:  {df.shape}")
df.head()


[DATA] Loaded: /Users/ravi/Desktop/Paper/hybrid-privacy-fl-healthcare/data/anonymized/Final_Depression1_anonymised.csv
[DATA] Shape:  (11670, 39)


,education,urban,gender,engnat,screensize,uniquenetworklocation,hand,religion,orientation,race,...,TIPI2,TIPI3,TIPI4,TIPI5,TIPI6,TIPI7,TIPI8,TIPI9,TIPI10,Condition
0,TKN_25ff27d8,TKN_2915511a,Female,Yes,TKN_ea75e325,TKN_bdb4a209,TKN_92b09c7c,TKN_05b81f76,TKN_1b5799db,TKN_25a81701,...,2,5,6,6,6,6,5,2,1,Extremely Severe
1,TKN_25ff27d8,TKN_2915511a,Female,Yes,TKN_ea75e325,TKN_9134e5db,TKN_92b09c7c,TKN_05b81f76,TKN_1b5799db,TKN_25a81701,...,2,5,7,7,5,5,5,3,5,Mild
2,TKN_25ff27d8,TKN_2915511a,Female,Yes,TKN_f1f2b172,TKN_9134e5db,TKN_92b09c7c,TKN_05b81f76,TKN_6b76bbae,TKN_25a81701,...,1,7,5,6,7,7,1,7,5,Mild
3,TKN_25ff27d8,TKN_2915511a,Female,Yes,TKN_ea75e325,TKN_9134e5db,TKN_92b09c7c,TKN_05b81f76,TKN_6b76bbae,TKN_25a81701,...,7,5,7,6,5,6,2,1,1,Severe
4,TKN_25ff27d8,TKN_2915511a,Female,Yes,TKN_f1f2b172,TKN_9134e5db,TKN_92b09c7c,TKN_05b81f76,TKN_6b76bbae,TKN_25a81701,...,4,3,6,4,6,4,5,3,3,Normal


In [3]:
# ── Cell 3: Preprocessing ─────────────────────────────────────
target_column = 'Condition'
df_encoded = df.copy()
label_encoders = {}

for col in df_encoded.select_dtypes(include='object').columns:
    if col != target_column:
        le = LabelEncoder()
        df_encoded[col] = le.fit_transform(df_encoded[col].astype(str))
        label_encoders[col] = le

le_target = LabelEncoder()
df_encoded[target_column] = le_target.fit_transform(df_encoded[target_column])

X = df_encoded.drop(columns=[target_column])
y = df_encoded[target_column]

print(f"[PREP] Features: {X.shape[1]}")
print(f"[PREP] Samples:  {X.shape[0]}")
print(f"[PREP] Classes:  {list(le_target.classes_)}")


[PREP] Features: 38
[PREP] Samples:  11670
[PREP] Classes:  ['Extremely Severe', 'Mild', 'Moderate', 'Normal', 'Severe']


In [4]:
# ── Cell 4: Train/Test Split ──────────────────────────────────
X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    stratify=y,
    test_size=0.2,
    random_state=SEED       # ← from config
)
print(f"[SPLIT] Train: {X_train.shape[0]}  Val: {X_val.shape[0]}")
print(f"[SPLIT] Class distribution (train):\n"
      f"{pd.Series(y_train).value_counts().sort_index()}")


[SPLIT] Train: 9336  Val: 2334
[SPLIT] Class distribution (train):
Condition
0    3174
1     880
2    1730
3    2045
4    1507
Name: count, dtype: int64


In [5]:
# ── Cell 5: SMOTE ─────────────────────────────────────────────
smote = SMOTE(random_state=SEED)    # ← from config
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print(f"[SMOTE] Before: {X_train.shape[0]}  After: {X_train_sm.shape[0]}")
print(f"[SMOTE] Class distribution after:\n"
      f"{pd.Series(y_train_sm).value_counts().sort_index()}")


[SMOTE] Before: 9336  After: 15870
[SMOTE] Class distribution after:
Condition
0    3174
1    3174
2    3174
3    3174
4    3174
Name: count, dtype: int64


In [6]:
# ── Cell 6: DataLoaders ───────────────────────────────────────
X_train_tensor = torch.tensor(X_train_sm.values, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train_sm,         dtype=torch.long)
X_val_tensor   = torch.tensor(X_val.values,       dtype=torch.float32)
y_val_tensor   = torch.tensor(y_val.values,       dtype=torch.long)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
val_dataset   = TensorDataset(X_val_tensor,   y_val_tensor)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False)

print(f"[DATA] Train batches: {len(train_loader)}")
print(f"[DATA] Val batches:   {len(val_loader)}")


[DATA] Train batches: 496
[DATA] Val batches:   73


In [7]:
# ── Cell 7: Model Definition ──────────────────────────────────
class MLP(nn.Module):
    def __init__(self, input_dim, num_classes):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, HIDDEN_DIM_1),
            nn.ReLU(),
            nn.Linear(HIDDEN_DIM_1, HIDDEN_DIM_2),
            nn.ReLU(),
            nn.Linear(HIDDEN_DIM_2, num_classes)
        )
    def forward(self, x):
        return self.model(x)

input_dim  = X_train_sm.shape[1]
num_classes = OUTPUT_DIM
model = MLP(input_dim, num_classes)

total_params = sum(p.numel() for p in model.parameters())
print(f"[MODEL] Input dim:  {input_dim}")
print(f"[MODEL] Architecture: {input_dim} → "
      f"{HIDDEN_DIM_1} → {HIDDEN_DIM_2} → {num_classes}")
print(f"[MODEL] Total parameters: {total_params}")


[MODEL] Input dim:  38
[MODEL] Architecture: 38 → 128 → 64 → 5
[MODEL] Total parameters: 13573


In [8]:
# ── Cell 8: Optimizer and Privacy Engine ──────────────────────
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
criterion = nn.CrossEntropyLoss()

if DP_ENABLED:
    privacy_engine = PrivacyEngine()
    model, optimizer, train_loader = privacy_engine.make_private(
        module=model,
        optimizer=optimizer,
        data_loader=train_loader,
        noise_multiplier=NOISE_MULTIPLIER,
        max_grad_norm=MAX_GRAD_NORM,
    )
    print(f"[DP] Privacy engine attached")
    print(f"[DP] Noise multiplier: {NOISE_MULTIPLIER}")
    print(f"[DP] Max grad norm:    {MAX_GRAD_NORM}")
else:
    privacy_engine = None
    print("[DP] Differential privacy DISABLED for this variant")


[DP] Differential privacy DISABLED for this variant


In [9]:
# ── Cell 9: Local Training ────────────────────────────────────
import time

best_val_loss = float('inf')
patience_counter = 0

for epoch in range(MAX_EPOCHS):
    start = time.time()

    # Train
    model.train()
    train_loss = 0.0
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        loss = criterion(model(X_batch), y_batch)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    # Validate
    model.eval()
    val_loss = 0.0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            out = model(X_batch)
            val_loss += criterion(out, y_batch).item()
            all_preds.extend(torch.argmax(out, dim=1).numpy())
            all_labels.extend(y_batch.numpy())

    val_acc = accuracy_score(all_labels, all_preds)
    elapsed = time.time() - start

    epsilon = (privacy_engine.get_epsilon(delta=DELTA)
               if DP_ENABLED else None)

    # Log every epoch
    log_round(
        client_id=CLIENT_ID,
        fl_round=epoch,
        val_accuracy=round(val_acc, 4),
        train_loss=round(train_loss, 4),
        val_loss=round(val_loss, 4),
        epsilon=round(epsilon, 4) if epsilon else None,
        elapsed_seconds=round(elapsed, 2)
    )

    eps_str = f" | ε={epsilon:.4f}" if epsilon else ""
    print(f"Epoch {epoch+1:02d} | "
          f"Train Loss: {train_loss:.4f} | "
          f"Val Loss: {val_loss:.4f} | "
          f"Val Acc: {val_acc:.4f}{eps_str}")

    # Early stopping on val_loss
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        torch.save(model.state_dict(), MODEL_FILE)
        print(f"  ✓ Best model saved → {MODEL_FILE}")
    else:
        patience_counter += 1
        if patience_counter >= EARLY_STOPPING_PATIENCE:
            print(f"[STOP] Early stopping at epoch {epoch+1}")
            break

if DP_ENABLED:
    final_epsilon = privacy_engine.get_epsilon(delta=DELTA)
    print(f"\n[DP] Final ε = {final_epsilon:.4f} at δ = {DELTA}")


[LOG] Saved → results/fl_anon_he.jsonl
Epoch 01 | Train Loss: 324.6692 | Val Loss: 32.9597 | Val Acc: 0.8029
  ✓ Best model saved → /Users/ravi/Desktop/Paper/hybrid-privacy-fl-healthcare/results/hospital_1_local_model.pth
[LOG] Saved → results/fl_anon_he.jsonl
Epoch 02 | Train Loss: 222.1255 | Val Loss: 35.8935 | Val Acc: 0.7806
[LOG] Saved → results/fl_anon_he.jsonl
Epoch 03 | Train Loss: 202.5373 | Val Loss: 26.2433 | Val Acc: 0.8415
  ✓ Best model saved → /Users/ravi/Desktop/Paper/hybrid-privacy-fl-healthcare/results/hospital_1_local_model.pth
[LOG] Saved → results/fl_anon_he.jsonl
Epoch 04 | Train Loss: 189.1279 | Val Loss: 36.4354 | Val Acc: 0.7823
[LOG] Saved → results/fl_anon_he.jsonl
Epoch 05 | Train Loss: 180.6217 | Val Loss: 25.3047 | Val Acc: 0.8500
  ✓ Best model saved → /Users/ravi/Desktop/Paper/hybrid-privacy-fl-healthcare/results/hospital_1_local_model.pth
[LOG] Saved → results/fl_anon_he.jsonl
Epoch 06 | Train Loss: 174.9329 | Val Loss: 27.1168 | Val Acc: 0.8299
[LOG] S

In [10]:
import os
from config import RESULTS_PATH

# Check 1: Log file created
log_path = os.path.join(RESULTS_PATH, "fl_anon_dp_he.jsonl")
print(f"✓ Log file exists: {os.path.exists(log_path)}")

# Check 2: Model file saved
model_path = os.path.join(RESULTS_PATH, "hospital_1_local_model.pth")
print(f"✓ Model saved: {os.path.exists(model_path)}")

# Check 3: Show last 2 log entries
if os.path.exists(log_path):
    import json
    with open(log_path) as f:
        lines = f.readlines()
    print(f"✓ Total log entries: {len(lines)}")
    print(f"  Last entry: {json.loads(lines[-1])['metrics']}")


✓ Log file exists: False
✓ Model saved: True


In [11]:
# ── Cell 10: CKKS Context + Helpers ──────────────────────────
import tenseal as ts
import pickle, os, requests, warnings
warnings.filterwarnings("ignore")

from config import (
    POLY_MODULUS_DEGREE, COEFF_MOD_BITS, GLOBAL_SCALE,
    CHUNK_SIZE, SERVER_URL, RESULTS_PATH
)

def get_ckks_context():
    ctx = ts.context(
        ts.SCHEME_TYPE.CKKS,
        poly_modulus_degree=POLY_MODULUS_DEGREE,
        coeff_mod_bit_sizes=COEFF_MOD_BITS
    )
    ctx.global_scale = GLOBAL_SCALE
    ctx.generate_galois_keys()
    return ctx

ckks_context    = get_ckks_context()
public_context  = ckks_context.serialize(save_secret_key=False)

# Save public context for server
context_path = os.path.join(RESULTS_PATH, "context.ser")
with open(context_path, "wb") as f:
    f.write(public_context)
print(f"[HE] CKKS context ready  (poly_mod={POLY_MODULUS_DEGREE})")
print(f"[HE] Public context saved → {context_path}")


def flatten_gradients(mdl):
    grads = [p.grad.view(-1).cpu()
             for p in mdl.parameters() if p.grad is not None]
    flat = torch.cat(grads)
    print(f"[HE] Flattened gradient length: {len(flat)}")
    return flat


def encrypt_gradients(flat_grads, pub_ctx):
    n_chunks = (len(flat_grads) + CHUNK_SIZE - 1) // CHUNK_SIZE
    chunks   = torch.chunk(flat_grads, n_chunks)
    ctx      = ts.context_from(pub_ctx)
    ctx.make_context_public()
    encrypted = []
    for i, chunk in enumerate(chunks):
        enc = ts.ckks_vector(ctx, chunk.tolist())
        encrypted.append(enc)
        print(f"[HE] Encrypted chunk {i} ({len(chunk)} values)")
    return encrypted


def send_encrypted_chunks(enc_chunks, url, client_id):
    for i, enc in enumerate(enc_chunks):
        payload = pickle.dumps({
            'chunk_id':  i,
            'client_id': client_id,
            'data':      enc.serialize()
        })
        try:
            r = requests.post(
                url, data=payload,
                headers={'Content-Type': 'application/octet-stream'},
                verify=False
            )
            status = "OK" if r.status_code == 200 else f"HTTP {r.status_code}"
            print(f"[HE] Sent chunk {i} ({len(payload)} bytes) → {status}")
        except Exception as e:
            print(f"[HE] Error sending chunk {i}: {e}")


def receive_and_apply_gradient(mdl, ctx, grad_file):
    with open(grad_file, "rb") as f:
        serialized_chunks = pickle.load(f)

    decrypted_parts = []
    for i, chunk_ser in enumerate(serialized_chunks):
        enc  = ts.ckks_vector_from(ctx, chunk_ser)
        dec  = torch.tensor(enc.decrypt(), dtype=torch.float32)
        decrypted_parts.append(dec)
        print(f"[HE] Decrypted chunk {i} ({len(dec)} values)")

    global_grad = torch.cat(decrypted_parts)
    print(f"[HE] Total gradient length: {len(global_grad)}")

    # Clip
    global_grad = torch.nan_to_num(global_grad, nan=0.0, posinf=0.0, neginf=0.0)
    norm = global_grad.norm()
    print(f"[HE] Decrypted gradient norm: {norm:.4f}")
    # Only clip if genuinely explosive — threshold based on actual gradient scale
    CLIP_NORM = 1000.0
    if norm > CLIP_NORM:
        global_grad = global_grad / norm * CLIP_NORM
        print(f"[HE] Gradient clipped from {norm:.2f} to {CLIP_NORM}")

    # Apply
    ptr = 0
    for param in mdl.parameters():
        if param.grad is not None:
            sz    = param.grad.numel()
            param.data -= global_grad[ptr:ptr+sz].view(param.grad.shape)
            ptr  += sz
    print("[HE] Global gradient applied to model")


def evaluate(mdl, loader, crit, split="Val"):
    mdl.eval()
    total_loss, preds, labels = 0.0, [], []
    with torch.no_grad():
        for xb, yb in loader:
            out = mdl(xb)
            total_loss += crit(out, yb).item()
            preds.extend(torch.argmax(out, dim=1).numpy())
            labels.extend(yb.numpy())
    acc = accuracy_score(labels, preds)
    print(f"[EVAL] {split} → Acc: {acc:.4f}  Loss: {total_loss:.4f}")
    return acc, total_loss, preds, labels


[HE] CKKS context ready  (poly_mod=8192)
[HE] Public context saved → /Users/ravi/Desktop/Paper/hybrid-privacy-fl-healthcare/results/context.ser


In [12]:
# CKKS sanity check
test_vector = [0.1, 0.2, 0.3, 0.4, 0.5]
ctx_test = ts.context_from(public_context)
ctx_test.make_context_public()
enc = ts.ckks_vector(ctx_test, test_vector)
dec = ts.ckks_vector_from(ckks_context, enc.serialize()).decrypt()
print(f"[HE] Original:  {test_vector}")
print(f"[HE] Decrypted: {[round(x,4) for x in dec[:5]]}")
print(f"[HE] ✓ CKKS working correctly")

# Check context file saved
context_path = os.path.join(RESULTS_PATH, "context.ser")
print(f"[HE] Context file saved: {os.path.exists(context_path)}")


[HE] Original:  [0.1, 0.2, 0.3, 0.4, 0.5]
[HE] Decrypted: [0.1, 0.2, 0.3, 0.4, 0.5]
[HE] ✓ CKKS working correctly
[HE] Context file saved: True


In [13]:
# ── Cell 11: Federated Learning Loop ─────────────────────────
import time, os, pickle, warnings, requests
warnings.filterwarnings("ignore")

from config import (
    RESULTS_PATH, SERVER_URL, FL_ROUNDS,
    HE_ENABLED, LEARNING_RATE, EARLY_STOPPING_PATIENCE,
    AGGREGATED_GRAD_FILE
)
from utils.logger import log_round
from sklearn.metrics import classification_report

# ── Use CLIENT_ID from Cell 1, AGGREGATED_GRAD_FILE from config ──
MODEL_FILE           = os.path.join(RESULTS_PATH, f"{CLIENT_ID}_local_model.pth")
GLOBAL_GRADIENT_FILE = AGGREGATED_GRAD_FILE   # ← config, never hardcoded

# ── Load best local model from Cell 9 training ───────────────
global_model = MLP(input_dim, num_classes)
state_dict   = torch.load(MODEL_FILE)
fixed_sd     = {k.replace("_module.", ""): v for k, v in state_dict.items()}
global_model.load_state_dict(fixed_sd, strict=False)
print(f"[FL] Loaded model from {MODEL_FILE}")

fl_criterion     = nn.CrossEntropyLoss()
best_val_loss    = float("inf")
patience_counter = 0

for fl_round in range(FL_ROUNDS):
    print(f"\n{'='*50}")
    print(f"  Federated Round {fl_round + 1} / {FL_ROUNDS}")
    print(f"{'='*50}")
    round_start = time.time()

    # STEP 1: One local training pass to update model weights
    fl_optimizer = torch.optim.Adam(global_model.parameters(), lr=LEARNING_RATE)
    global_model.train()
    for xb, yb in train_loader:
        fl_optimizer.zero_grad()
        loss = fl_criterion(global_model(xb), yb)
        loss.backward()
        fl_optimizer.step()

    # STEP 2: Compute gradient for transmission
    # Must do a fresh forward+backward to populate param.grad before flattening
    grad_optimizer = torch.optim.Adam(global_model.parameters(), lr=LEARNING_RATE)
    grad_optimizer.zero_grad()
    global_model.train()
    n_batches = 0
    for xb, yb in train_loader:
        loss = fl_criterion(global_model(xb), yb)
        loss.backward()
        n_batches += 1
    # Average gradients over batches so norm is comparable to single-batch scale
    with torch.no_grad():
        for param in global_model.parameters():
            if param.grad is not None:
                param.grad /= n_batches
    grads = flatten_gradients(global_model)
    print(f"[FL] Gradient norm: {grads.norm():.4f}  length: {len(grads)}  batches: {n_batches}")
    
    # STEP 3: Send gradients to server
    if HE_ENABLED:
        enc_grads = encrypt_gradients(grads, public_context)
        send_encrypted_chunks(enc_grads, SERVER_URL, CLIENT_ID)
    else:
        r = requests.post(
            SERVER_URL + "/raw",
            json={"client_id": CLIENT_ID, "gradients": grads.tolist()},
            verify=False
        )
        print(f"[FL] Sent raw gradients → HTTP {r.status_code}")

    # STEP 4: Wait for Aggregate.ipynb to write the global gradient file
    print("[FL] Waiting for aggregated gradient file...")
    timed_out = False
    wait_start = time.time()
    while not os.path.exists(GLOBAL_GRADIENT_FILE):
        time.sleep(2)
        if time.time() - wait_start > 300:
            print("[FL] Timeout — Aggregate.ipynb did not run within 5 min.")
            timed_out = True
            break

    if timed_out:
        print(f"[FL] Skipping round {fl_round + 1} due to timeout.")
        continue   # ← skip to next round, do not try to open file

    # STEP 5: Apply aggregated gradient
    receive_and_apply_gradient(global_model, ckks_context, GLOBAL_GRADIENT_FILE)

    # STEP 6: Evaluate
    val_acc, val_loss, val_preds, val_labels = evaluate(
        global_model, val_loader, fl_criterion, "Val"
    )
    train_acc, train_loss, _, _ = evaluate(
        global_model, train_loader, fl_criterion, "Train"
    )

    elapsed = time.time() - round_start
    print(f"[FL] Round {fl_round+1} complete in {elapsed:.1f}s")
    print(classification_report(val_labels, val_preds, zero_division=0))

    # STEP 7: Log
    log_round(
        client_id=CLIENT_ID,
        fl_round=fl_round + 1,
        val_accuracy=round(val_acc, 4),
        train_loss=round(train_loss, 4),
        val_loss=round(val_loss, 4),
        epsilon=None,
        elapsed_seconds=round(elapsed, 2)
    )

    # STEP 8: Early stopping and best model save
    if val_loss < best_val_loss:
        best_val_loss    = val_loss
        patience_counter = 0
        save_path = os.path.join(RESULTS_PATH, f"{CLIENT_ID}_global_model.pth")
        torch.save(global_model.state_dict(), save_path)
        print(f"[FL] ✓ Best global model saved → {save_path}")
    else:
        patience_counter += 1
        if patience_counter >= EARLY_STOPPING_PATIENCE:
            print(f"[FL] Early stopping at round {fl_round+1}")
            break

    # STEP 9: Delete local copy of aggregated file so next round polls fresh
    # NOTE: Do NOT delete this if other clients may not have read it yet.
    # Only safe to delete after all clients confirm receipt — left to Aggregate.
    # Aggregate.ipynb will overwrite this file each round automatically.

print(f"\n[FL] Training complete. Best val loss: {best_val_loss:.4f}")


[FL] Loaded model from /Users/ravi/Desktop/Paper/hybrid-privacy-fl-healthcare/results/hospital_1_local_model.pth

  Federated Round 1 / 10
[HE] Flattened gradient length: 13573
[FL] Gradient norm: 3.5491  length: 13573  batches: 496
The following operations are disabled in this setup: matmul, matmul_plain, enc_matmul_plain, conv2d_im2col.
If you need to use those operations, try increasing the poly_modulus parameter, to fit your input.
[HE] Encrypted chunk 0 (6787 values)
The following operations are disabled in this setup: matmul, matmul_plain, enc_matmul_plain, conv2d_im2col.
If you need to use those operations, try increasing the poly_modulus parameter, to fit your input.
[HE] Encrypted chunk 1 (6786 values)
[HE] Sent chunk 0 (668591 bytes) → OK
[HE] Sent chunk 1 (669016 bytes) → OK
[FL] Waiting for aggregated gradient file...
[HE] Decrypted chunk 0 (6787 values)
[HE] Decrypted chunk 1 (6786 values)
[HE] Total gradient length: 13573
[HE] Decrypted gradient norm: inf
[HE] Gradient cl

In [ ]:
from utils.logger import log_result

log_result(
    experiment_name="ablation_full_framework",
    client_id="hospital_1",
    metrics={
        "fl_round": round_num,
        "val_accuracy": val_acc,
        "epsilon": epsilon,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "training_time_seconds": elapsed
    }
)


In [ ]:
# Simulate a dummy gradient from a layer (e.g., 5 parameters)
local_gradient = torch.tensor([0.25, -0.3, 0.1, 0.55, -0.12], dtype=torch.float32)

print(" Original Gradient (Local Model):")
print(local_gradient.numpy())


In [ ]:
# import torch
# import tenseal as ts
# import numpy as np

# def print_raw_gradients(model):
#     print("\n Raw Gradients Before Encryption:")
#     for name, param in model.named_parameters():
#         if param.requires_grad and param.grad is not None:
#             grad = param.grad.view(-1).detach().cpu().numpy()
#             print(f"\n Layer: {name}")
#             print(f"   Gradient Shape: {grad.shape}")
#             print(f"   Sample Values: {grad[:5]}")
#             print("-" * 50)

# def encrypt_gradients_from_model(model, X_batch, y_batch, loss_fn, context):
#     model.train()
#     model.zero_grad()

#     # Forward + Backward Pass
#     outputs = model(X_batch)
#     loss = loss_fn(outputs, y_batch)
#     loss.backward()  # Only once!

#     #  Print raw gradients before encryption
#     print_raw_gradients(model)

#     encrypted_grads = {}

#     for name, param in model.named_parameters():
#         if param.requires_grad and param.grad is not None:
#             grad = param.grad.view(-1).detach().cpu().numpy()
#             enc_grad = ts.ckks_vector(context, grad.tolist())
#             encrypted_grads[name] = enc_grad

#             print(f"\n Encrypted Gradient for {name} — sample decrypted view:")
#             print(enc_grad.decrypt()[:5])

#     return encrypted_grads
# # Example call (make sure this is included below the function definition)
# encrypted_grads = encrypt_gradients_from_model(model, X_batch, y_batch, loss_fn, public_context)



In [ ]:
# CKKS encryption context setup
context = ts.context(
    ts.SCHEME_TYPE.CKKS,
    poly_modulus_degree=8192,
    coeff_mod_bit_sizes=[60, 40, 40, 60]
)
context.generate_galois_keys()
context.global_scale = 2**40

# Save public & secret key (just for simulation; normally private key stays secure)
public_context = context.copy()  # This is what would be shared with the server
public_context.make_context_public()  # Only allows encryption
with open("context.ser", "wb") as f:
    f.write(public_context.serialize())


In [ ]:
import torch

# Set device (GPU if available)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [ ]:
def encrypt_gradients_from_model(model, X_batch, y_batch, loss_fn, context):
    model.train()
    model.zero_grad()

    outputs = model(X_batch)
    loss = loss_fn(outputs, y_batch)
    loss.backward()

    encrypted_grads = {}

    for name, param in model.named_parameters():
        if param.requires_grad and param.grad is not None:
            grad = param.grad.view(-1).detach().cpu().numpy()
            enc_grad = ts.ckks_vector(context, grad.tolist())
            encrypted_grads[name] = enc_grad

            print(f"\n Layer: {name}")
            print(f"   Original Gradient Sample: {grad[:5]}")
            print(f"   Encrypted Gradient Sample (decrypted view): {enc_grad.decrypt()[:5]}")

    return encrypted_grads


In [ ]:
import torch.nn as nn

loss_fn = nn.CrossEntropyLoss()

# Get one batch from your train loader
X_sample, y_sample = next(iter(train_loader))
X_sample, y_sample = X_sample.to(device), y_sample.to(device)

# Run encryption on gradients
encrypted_grads = encrypt_gradients_from_model(model, X_sample, y_sample, loss_fn, context)


In [ ]:
import torch
import tenseal as ts
import pickle
import requests
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

# -------------------------------
# CKKS Setup 
# -------------------------------
def get_ckks_context():
    context = ts.context(
        ts.SCHEME_TYPE.CKKS,
        poly_modulus_degree=8192,
        coeff_mod_bit_sizes=[60, 40, 40, 60]
    )
    context.global_scale = 2**40
    context.generate_galois_keys()
    return context

ckks_context = get_ckks_context()
public_context = ckks_context.serialize(save_secret_key=False)

# -------------------------------
# Gradient Flattening
# -------------------------------
def flatten_gradients(model):
    grads = []
    for param in model.parameters():
        if param.grad is not None:
            grads.append(param.grad.view(-1).cpu())
    flat = torch.cat(grads)
    print(f"[DEBUG] Flattened gradient length: {len(flat)}")
    return flat

# # ------------------------------------------
# # Inject Byzantine Gradient Manipulation
# # ------------------------------------------
# def make_byzantine_gradients(grads, strategy="flip"):
#     if strategy == "flip":
#         print("[BYZANTINE] Flipping gradients (multiply by -1)...")
#         return -1 * grads
#     elif strategy == "scale":
#         print("[BYZANTINE] Scaling gradients by 100...")
#         return grads * 100
#     elif strategy == "random":
#         print("[BYZANTINE] Replacing with random noise...")
#         return torch.randn_like(grads)
#     else:
#         print("[BYZANTINE] Unknown strategy, sending original gradients.")
#         return grads

# -------------------------------
# Encrypt Gradients with CKKS
# -------------------------------
def encrypt_gradients(flattened_grads, context, chunk_size=8000):
    if len(flattened_grads) > chunk_size:
        print("[WARNING] Gradient too large for single CKKS vector (>8192). Chunking.")
        chunks = torch.chunk(flattened_grads, (len(flattened_grads) + chunk_size - 1) // chunk_size)
    else:
        chunks = [flattened_grads]

    encrypted_chunks = []
    ctx = ts.context_from(context)
    ctx.make_context_public()

    for i, chunk in enumerate(chunks):
        enc = ts.ckks_vector(ctx, chunk.tolist())
        encrypted_chunks.append(enc)
        print(f"[DEBUG] Encrypted chunk {i} with {len(chunk)} values")

    return encrypted_chunks

# -------------------------------
# Send Encrypted Chunks to Server
# -------------------------------
def send_encrypted_chunks_binary(encrypted_chunks, server_url, client_id="hospital_1"):
    for i, enc_chunk in enumerate(encrypted_chunks):
        try:
            payload = {
                'chunk_id': i,
                'client_id': client_id, 
                'data': enc_chunk.serialize()
            }
            data = pickle.dumps(payload)

            response = requests.post(
                server_url,
                data=data,
                headers={'Content-Type': 'application/octet-stream'},
                verify=False  
            )
            if response.status_code == 200:
                print(f"[CLIENT] Sent chunk {i} ({len(data)} bytes)")
            else:
                print(f"[CLIENT] Failed to send chunk {i}: HTTP {response.status_code}")
        except Exception as e:
            print(f"[CLIENT]  Error sending chunk {i}: {e}")


# -------------------------------
# Client Main Routine
# -------------------------------
def run_client_post_training(model):
    print("[CLIENT] Preparing gradients...")
    grads = flatten_gradients(model)
    enc_chunks = encrypt_gradients(grads, public_context)

    print("[CLIENT] Sending encrypted gradients...")
    send_encrypted_chunks_binary(enc_chunks, server_url="https://127.0.0.1:5055")

run_client_post_training(model)


In [ ]:
import torch
import torch.nn as nn
import pickle
import tenseal as ts
from sklearn.metrics import classification_report
import warnings
import os
import shutil
import requests

warnings.filterwarnings("ignore", category=UserWarning)

# -------------------------------
# Define MLP Architecture
# -------------------------------
class MLP(nn.Module):
    def __init__(self, input_dim, hidden_dim1=128, hidden_dim2=64, output_dim=5):
        super(MLP, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, hidden_dim1),
            nn.ReLU(),
            nn.Linear(hidden_dim1, hidden_dim2),
            nn.ReLU(),
            nn.Linear(hidden_dim2, output_dim)
        )

    def forward(self, x):
        return self.model(x)

# -------------------------------
# Decrypt + Apply Global Gradient
# -------------------------------
def receive_and_update_model(model, ckks_context, global_gradient_file="aggregated_gradient_global_encrypted.pkl"):
    with open(global_gradient_file, "rb") as f:
        encrypted_chunks_serialized = pickle.load(f)

    decrypted_flattened_grad = []
    for i, chunk_ser in enumerate(encrypted_chunks_serialized):
        enc_chunk = ts.ckks_vector_from(ckks_context, chunk_ser)
        decrypted_chunk = torch.tensor(enc_chunk.decrypt(), dtype=torch.float32)
        decrypted_flattened_grad.append(decrypted_chunk)
        print(f"[CLIENT] Decrypted chunk {i} of size {len(decrypted_chunk)}")

    decrypted_gradient = torch.cat(decrypted_flattened_grad)
    print(f"[CLIENT] Total decrypted global gradient length: {len(decrypted_gradient)}")

    # Clip gradient norm if necessary
    grad_norm = decrypted_gradient.norm()
    if grad_norm > 1e2:
        decrypted_gradient = decrypted_gradient / grad_norm * 1e2
        print(f"[CLIENT] Gradient clipped to norm 1e2 (original norm: {grad_norm:.2e})")

    # Apply gradients to model
    pointer = 0
    for param in model.parameters():
        if param.grad is not None:
            param_shape = param.grad.shape
            param_size = param.grad.numel()
            grad_slice = decrypted_gradient[pointer:pointer + param_size].view(param_shape)
            param.data -= grad_slice
            pointer += param_size

    print("[CLIENT] Global model updated with aggregated global gradient")

# -------------------------------
# Evaluate Model
# -------------------------------
def evaluate_model(model, dataloader, criterion, split_name="Validation"):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for x_batch, y_batch in dataloader:
            outputs = model(x_batch)
            loss = criterion(outputs, y_batch)
            total_loss += loss.item()
            preds = torch.argmax(outputs, dim=1)
            correct += (preds == y_batch).sum().item()
            total += y_batch.size(0)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y_batch.cpu().numpy())

    acc = 100.0 * correct / total
    print(f"[CLIENT] {split_name} → Accuracy: {acc:.2f}%, Loss: {total_loss:.4f}")
    return all_preds, all_labels, total_loss

# -------------------------------
# HE Gradient Send Helpers
# -------------------------------
def flatten_gradients(model):
    grads = []
    for param in model.parameters():
        if param.grad is not None:
            grads.append(param.grad.view(-1).cpu())
    flat = torch.cat(grads)
    return flat
    
def encrypt_gradients(flattened_grads, context, chunk_size=8000):
    chunks = torch.chunk(flattened_grads, (len(flattened_grads) + chunk_size - 1) // chunk_size)
    encrypted_chunks = []
    ctx = ts.context_from(context)
    ctx.make_context_public()

    for i, chunk in enumerate(chunks):
        enc = ts.ckks_vector(ctx, chunk.tolist())
        encrypted_chunks.append(enc)
        print(f"[DEBUG] Encrypted chunk {i} with {len(chunk)} values")
    return encrypted_chunks

def send_encrypted_chunks_binary(encrypted_chunks, server_url, client_id="hospital_1"):
    for i, enc_chunk in enumerate(encrypted_chunks):
        try:
            payload = {
                'chunk_id': i,
                'client_id': client_id,
                'data': enc_chunk.serialize()
            }
            data = pickle.dumps(payload)
            response = requests.post(
                server_url,
                data=data,
                headers={'Content-Type': 'application/octet-stream'},
                verify=False
            )
            if response.status_code == 200:
                print(f"[CLIENT] Sent chunk {i} ({len(data)} bytes)")
            else:
                print(f"[CLIENT] Failed to send chunk {i}: HTTP {response.status_code}")
        except Exception as e:
            print(f"[CLIENT] Error sending chunk {i}: {e}")

# -------------------------------
# Main Federated Loop (Last Client)
# -------------------------------
ckks_context = ts.context(ts.SCHEME_TYPE.CKKS, poly_modulus_degree=8192, coeff_mod_bit_sizes=[60, 40, 40, 60])
ckks_context.global_scale = 2**40
ckks_context.generate_galois_keys()
public_context = ckks_context.serialize(save_secret_key=False)

# Init model
sample_batch, _ = next(iter(train_loader))
input_dim = sample_batch.shape[1]
global_model = MLP(input_dim=input_dim, output_dim=5)
state_dict = torch.load("trained_mlp_model.pth")
fixed_state_dict = {k.replace("_module.", ""): v for k, v in state_dict.items()}
global_model.load_state_dict(fixed_state_dict, strict=False)
print("[CLIENT] Loaded local model weights from 'trained_mlp_model.pth'")

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(global_model.parameters(), lr=0.01)

best_val_loss = float('inf')
patience = 3
counter = 0
max_rounds = 50

for round in range(max_rounds):
    print(f"\n === Federated Round {round+1} ===")

    # STEP 1: Local training
    global_model.train()
    for x_batch, y_batch in train_loader:
        optimizer.zero_grad()
        outputs = global_model(x_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()

    # STEP 2: Wait for global gradient
    user_in = input("Press Enter once server has aggregated and saved encrypted global gradient (or type 'exit'): ").strip().lower()
    if user_in == "exit":
        print("Exiting federated training loop.")
        break

    # STEP 3: Apply global gradient
    receive_and_update_model(global_model, ckks_context)

    # STEP 4: Re-train model (after update)
    global_model.train()
    for x_batch, y_batch in train_loader:
        optimizer.zero_grad()
        outputs = global_model(x_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()

    # STEP 5: Send new gradients
    print("[CLIENT] Encrypting & sending updated gradients...")
    grads = flatten_gradients(global_model)
    encrypted_chunks = encrypt_gradients(grads, public_context)
    send_encrypted_chunks_binary(encrypted_chunks, server_url="https://127.0.0.1:5055", client_id="hospital_1")

    # STEP 6: Evaluation
    evaluate_model(global_model, train_loader, criterion, "Training")
    val_preds, val_labels, val_loss = evaluate_model(global_model, val_loader, criterion, "Validation")
    print("[CLIENT] Validation Classification Report:")
    print(classification_report(val_labels, val_preds, zero_division=0))

    # STEP 7: Early stopping
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        counter = 0
        torch.save(global_model.state_dict(), "trained_global_model_1.pth")
        print("Best model saved.")
    else:
        counter += 1
        if counter >= patience:
            print("Early stopping.")
            break
